In [ ]:
# Bibliotecas importadas
import os
import re
import pandas as pd
import pyodbc as odbc

# Pega usuário de rede
usuario = os.getenv("USERNAME")

# Conexão com o DW
path = fr'CAMINHO DO ARQUIVO COM O LOGIN E SENHA'
os.chdir(path)
df_senhas = pd.read_excel('NOME_DO_ARQUIVO.xlsx')
server = df_senhas.iloc[0,0]
database = df_senhas.iloc[0,1]
username = df_senhas.iloc[0,2]
password = df_senhas.iloc[0,3]
conn = odbc.connect(f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}')

In [ ]:
# Base 1 - Cálculo de Inadimplência
# Seleciona a última data da base -> ela não entra na análise
script = """
select max(Data_Ref) from Base_Historica
"""
df = pd.read_sql(script, conn)
data_retirar = str(df.iloc[0,0])

script = """
SELECT
    a.Data_Ref,
    a.Data_Aloc,
    a.ID_Cota,
    a.Macrosituacao,
    b.Sit_Contrato,
    b.Data_Cancel,
    a.Grana_Total
FROM Base_Historica a
    left join Alocacoes b on a.ID_Cota = b.ID_Cota
where a.Data_Ref != ?
and a.Data_Aloc >= '2024-01-01'
"""
# Carregando a base
df = pd.read_sql(script, conn, params = [data_retirar])

# Base 2 - Cálculo de Cancelamento
script = """
select
    a.ID_Cota,
    a.Data_Aloc,
    a.Data_Cancel,
    a.Sit_Contrato,
    a.Grana_Total
from Alocacoes a
    where a.Data_Aloc >= '2024-01-01'
"""
df2 = pd.read_sql(script, conn)

# Base 3 - Motivo Cancelamento
script = """
select
    a.ID_Cota,
    a.Data_Cancel,
    a.Motivo_Cancelamento
from Cancelamento a
"""
df3 = pd.read_sql(script, conn)

# União do df2 e df3
df2 = df2.merge(df3, how='left', on=['ID_Cota', 'Data_Cancel'])
x1 = df2[df2['Motivo_Cancelamento'].isna()]
x2 = df2[df2['Motivo_Cancelamento'] == 'INADIMPLÊNCIA']
df2 = pd.concat([x1, x2])
df2 = df2.drop_duplicates(subset=['ID_Cota'])

C:\Users\GabrielHenriqueSilva\AppData\Local\Temp\ipykernel_28900\3192489916.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(script, conn)
C:\Users\GabrielHenriqueSilva\AppData\Local\Temp\ipykernel_28900\3192489916.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(script, conn, params = [data_retirar])
C:\Users\GabrielHenriqueSilva\AppData\Local\Temp\ipykernel_28900\3192489916.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql(script, conn)
C:\Users\GabrielHenrique

In [ ]:
# Análise de Inadimplência por Safra
df_final = df.copy()

# Convertendo as colunas para datetime
df_final['Data_Ref'] = pd.to_datetime(df_final['Data_Ref'])
df_final['Data_Aloc'] = pd.to_datetime(df_final['Data_Aloc'])

# Criando uma coluna de ano-mês para 'DT_Alocacao' e 'DT_Referencia'
df_final['Ano_Mes_Alocacao'] = df_final['Data_Aloc'].dt.to_period('M')
df_final['Ano_Mes_Referencia'] = df_final['Data_Aloc'].dt.to_period('M')

# Vamos agora calcular o percentual de inadimplentes para cada combinação
# Primeiramente, criaremos uma coluna para marcar se está "Inadimplente"
df_final['Inadimplente'] = df_final['Macrosituacao'] == 'Inadimplente'

# Função para calcular a diferença em meses entre 'DT_Alocacao' e 'DT_Referencia'
def calcular_mes_diff(row):
    return (row['Data_Ref'].year - row['Data_Aloc'].year) * 12 + row['Data_Ref'].month - row['Data_Aloc'].month

# Adiciona a coluna com a diferença em meses
df_final['Mes_Diff'] = df_final.apply(calcular_mes_diff, axis=1)

# Agora vamos criar a matriz
# A ideia é calcular, para cada 'DT_Alocacao', o percentual de inadimplência para cada 'DT_Referencia'

# Matriz de percentual de inadimplentes por mês de DT_Alocacao e DT_Referencia
matriz = pd.pivot_table(df_final, 
                       values='Grana_Total', 
                       index='Ano_Mes_Alocacao', 
                       columns='Mes_Diff', 
                       aggfunc=lambda x: (x[df_final['Inadimplente']].sum() / x.sum()) * 100)

# Arredondando para duas casas decimais
matriz = matriz.round(1)

# Retirando M0
#matriz = matriz.drop(columns=['M0'])

# Renomeando as colunas para 'M1', 'M2', 'M3', etc.
matriz.columns = ['M' + str(i) for i in matriz.columns]

# Retirando o index
matriz = matriz.reset_index()

# Substituindo NaN por "-"
matriz = matriz.fillna('-')

# Retirando coluna M0
matriz = matriz.drop(columns=['M0'])

# Tratando coluna final
matriz = matriz.rename(columns = {'Ano_Mes_Alocacao': 'Safra Alocação'})

In [33]:
matriz

,Safra Alocação,M1,M2,M3,M4,M5,M6,M7,M8,M9,M10,M11,M12,M13,M14
0,2024-01,4.0,14.7,20.7,20.8,15.6,14.9,15.1,14.5,13.4,12.4,12.2,12.4,13.6,13.2
1,2024-02,4.6,17.0,20.4,21.3,16.5,15.0,15.2,12.8,13.6,15.0,14.9,15.2,15.0,-
2,2024-03,4.1,11.5,14.1,15.1,13.2,14.0,13.5,13.9,13.9,12.3,12.8,12.7,-,-
3,2024-04,4.0,11.8,16.7,17.5,17.1,17.7,16.4,16.0,14.8,15.3,14.6,-,-,-
4,2024-05,4.7,11.8,15.6,16.1,15.3,14.8,14.9,13.9,14.3,12.4,-,-,-,-
5,2024-06,3.6,12.4,15.0,17.2,15.6,15.0,15.1,14.8,14.2,-,-,-,-,-
6,2024-07,2.9,9.7,16.0,16.8,16.3,15.5,16.9,15.0,-,-,-,-,-,-
7,2024-08,2.9,10.6,14.3,15.0,15.1,15.4,13.8,-,-,-,-,-,-,-
8,2024-09,2.3,11.8,14.8,16.1,15.3,14.9,-,-,-,-,-,-,-,-
9,2024-10,2.6,6.1,14.0,17.7,17.4,-,-,-,-,-,-,-,-,-
